# SIR–UKF estimation (logit / ALR reparameterization)

Cleaned, de-duplicated pipeline. There is now a single SIR transition and a single
sigma-point implementation, shared by the likelihood. Estimates are computed **once**
and cached to `cache_logit/`; reruns load the cached result instead of re-optimizing.

## 0. Imports and clean series

In [ ]:
import os, json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

weekly_all = pd.read_csv('weekly_all.csv', index_col=0, parse_dates=True)
daily_all  = pd.read_csv('daily_all.csv',  index_col=0, parse_dates=True)

# Convenience arrays (model always uses these names)
gt_weekly   = weekly_all['gtrends'].values
art_weekly  = weekly_all['neg_art_count'].values
tone_weekly = weekly_all['avg_tone'].values

gt_daily    = daily_all['gtrends'].values
art_daily   = daily_all['neg_art_count'].values
tone_daily  = daily_all['avg_tone'].values

## 1. Caching / IO helpers

`fit_or_load` runs the fit the first time and pickles the result; later runs load it.
`save_est_json` writes a human-readable copy of the estimates.

In [ ]:
CACHE_DIR = 'cache_logit'
FIG_DIR   = 'figures_logit'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

def fit_or_load(name, fit_fn, *args, recompute=False, **kwargs):
    path = f'{CACHE_DIR}/{name}.pkl'
    if (not recompute) and os.path.exists(path):
        with open(path, 'rb') as fh:
            print(f'[cache] loaded {path}')
            return pickle.load(fh)
    result = fit_fn(*args, **kwargs)
    with open(path, 'wb') as fh:
        pickle.dump(result, fh)
    print(f'[cache] saved {path}')
    return result

def save_est_json(name, est):
    with open(f'{CACHE_DIR}/est_{name}.json', 'w') as f:
        json.dump(est, f, indent=2,
                  default=lambda o: float(o) if hasattr(o, 'item') else o)

def savefig(fig, name, dpi=150):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=dpi, bbox_inches='tight')

## 2. Model building blocks

Additive log-ratio transform `(S, I) ↔ z` (keeps states on the simplex), one vectorized
SIR transition, and one Merwe scaled sigma-point routine.

In [ ]:
# Additive log-ratio transform
def _alr(S, I):
    R = max(1.0 - S - I, 1e-12)
    return np.array([np.log(max(S, 1e-12) / R), np.log(max(I, 1e-12) / R)])

def _inv_alr_rows(Z):            # (k,2) z -> (k,2) [S,I]
    Z = np.clip(Z, -30, 30)
    e = np.exp(Z)
    d = 1.0 + e[:, 0] + e[:, 1]
    return np.column_stack([e[:, 0] / d, e[:, 1] / d])

def _alr_rows(SI):               # (k,2) [S,I] -> (k,2) z
    S, I = SI[:, 0], SI[:, 1]
    R = np.clip(1.0 - S - I, 1e-12, None)
    return np.column_stack([np.log(np.clip(S, 1e-12, None) / R),
                            np.log(np.clip(I, 1e-12, None) / R)])

In [ ]:
# SIR transition (Euler), vectorized over sigma-point rows
def sir_transition_vec(X, beta, gamma, dt=1.0):
    """X = (k, 2) rows of [S, I] fractions. Returns (k, 2) [S_next, I_next]."""
    S, I = X[:, 0], X[:, 1]
    return np.column_stack([S - beta * S * I * dt,
                            I + (beta * S * I - gamma * I) * dt])

In [ ]:
# Merwe scaled sigma points: weights (computed once) + per-step sigma builder
def merwe_weights(n, alpha=1e-3, beta_p=2.0, kappa=0.0):
    lam = alpha**2 * (n + kappa) - n
    c   = n + lam
    Wm    = np.full(2 * n + 1, 0.5 / c); Wm[0] = lam / c
    Wc    = Wm.copy();                   Wc[0] += (1.0 - alpha**2 + beta_p)
    return Wm, Wc, c

def make_sigmas(x, P, c):
    """Returns (2n+1, n) sigma points; raises LinAlgError if P is not PD."""
    n = len(x)
    try:
        L = np.linalg.cholesky(c * P)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(c * P + 1e-9 * np.eye(n))
    return np.vstack([x, x + L.T, x - L.T])

## 3. Negative log-likelihood (3 all-interval channels, logit/ALR state)

Observation model per channel `k`: `y_k = coef_k * I + b_k`, with `coef = [1, c2, c3]`
and `b = [a1, a2, a3]`. Channels are masked per time step via `masks`.

In [ ]:
def negloglik_3d_allint_logit(params, observations, masks, dt=1.0):
    PENALTY = 1e10
    try:
        (log_g, log_R0, log_c2, log_negc3, a1, log_a2, a3,
         log_s2S, log_s2I, log_s2e1, log_s2e2, log_s2e3, logit_I0) = params
        gamma = np.exp(log_g); R0 = np.exp(log_R0); beta = gamma * R0
        c2 = np.exp(log_c2); c3 = -np.exp(log_negc3); a2 = np.exp(log_a2)
        Q = np.diag([np.exp(log_s2S), np.exp(log_s2I)])
        R = np.diag([np.exp(log_s2e1), np.exp(log_s2e2), np.exp(log_s2e3)])
        I0 = 1.0 / (1.0 + np.exp(-logit_I0))
        x = _alr(1.0 - I0, I0)
        P = np.diag([1e-4, 1e-4])
        Wm, Wc, cw = merwe_weights(2)
        coef = np.array([1.0, c2, c3]); b = np.array([a1, a2, a3])
        ll = 0.0
        for t in range(len(observations)):
            # --- predict ---
            sp   = make_sigmas(x, P, cw)
            SI   = _inv_alr_rows(sp)
            SI_f = sir_transition_vec(SI, beta, gamma, dt)
            sp_f = _alr_rows(SI_f)
            x = Wm @ sp_f; d = sp_f - x; P = (d.T * Wc) @ d + Q
            # --- update (masked channels only) ---
            idx = np.where(masks[t])[0]
            if len(idx) == 0:
                continue
            sp     = make_sigmas(x, P, cw)
            I_sp   = _inv_alr_rows(sp)[:, 1]
            sp_h   = I_sp[:, None] * coef[idx] + b[idx]
            y_pred = Wm @ sp_h; dy = sp_h - y_pred; dx = sp - x
            S_yy = (dy.T * Wc) @ dy + R[np.ix_(idx, idx)]
            P_xy = (dx.T * Wc) @ dy
            sign, logdet = np.linalg.slogdet(S_yy)
            if sign <= 0 or not np.isfinite(logdet):
                return PENALTY
            K  = P_xy @ np.linalg.inv(S_yy)
            nu = observations[t, idx] - y_pred
            x = x + K @ nu; P = P - K @ S_yy @ K.T
            ll += -0.5 * (len(nu) * np.log(2 * np.pi) + logdet
                          + nu @ np.linalg.solve(S_yy, nu))
        return -ll if np.isfinite(ll) else PENALTY
    except Exception:
        return PENALTY

## 4. Fit once and cache the estimates

The optimizer runs only on the first execution; afterwards the cached result is loaded
from `cache_logit/sir_ukf_weekly.pkl`. Set `recompute=True` to force a refit.

> Adjust `init_params` / the observation construction to match your data scaling —
> channel 0 is anchored to `I` with slope fixed at 1.

In [ ]:
# Stack the three weekly channels; mask non-finite entries so they are skipped.
observations = np.column_stack([gt_weekly, art_weekly, tone_weekly]).astype(float)
masks = np.isfinite(observations)
observations = np.nan_to_num(observations, nan=0.0)

PARAM_NAMES = ['log_g', 'log_R0', 'log_c2', 'log_negc3', 'a1', 'log_a2', 'a3',
               'log_s2S', 'log_s2I', 'log_s2e1', 'log_s2e2', 'log_s2e3', 'logit_I0']

init_params = np.array([
    np.log(0.10),                    # log_g     (gamma ~ 0.10)
    np.log(2.0),                     # log_R0    (R0 ~ 2.0)
    0.0,                             # log_c2    (c2 ~ 1)
    0.0,                             # log_negc3 (c3 ~ -1)
    float(np.nanmean(gt_weekly)),    # a1
    0.0,                             # log_a2    (a2 ~ 1)
    float(np.nanmean(tone_weekly)),  # a3
    np.log(1e-4),                    # log_s2S
    np.log(1e-4),                    # log_s2I
    np.log(1.0),                     # log_s2e1
    np.log(1.0),                     # log_s2e2
    np.log(1.0),                     # log_s2e3
    np.log(0.01 / 0.99),             # logit_I0  (I0 ~ 1%)
])

def fit_weekly():
    t0 = time.time()
    res = minimize(negloglik_3d_allint_logit, init_params,
                   args=(observations, masks), method='Nelder-Mead',
                   options={'maxiter': 20000, 'xatol': 1e-6, 'fatol': 1e-6})
    print(f'optimized in {time.time() - t0:.1f}s')
    return res

res = fit_or_load('sir_ukf_weekly', fit_weekly)

est = dict(zip(PARAM_NAMES, [float(v) for v in res.x]))
est['neg_loglik'] = float(res.fun)
save_est_json('sir_ukf_weekly', est)

print('neg-loglik =', res.fun)
for k, v in est.items():
    print(f'  {k:>10} = {v: .6f}')